# Tasks

Event loop напрямую с корутинами даже не умеет работать.

Но как же тогда запускать наши корутины в цикле событий?
Ответ: Надо “обернуть” их в объект класса asyncio.Task. Так как в цикле событий “крутятся” именно объекты Task, а не корутины напрямую.

Event loop оперирует  именно задачами (asyncio.Task), которые в свою очередь в себе содержат корутину, код которой и выполняется конкурентно в asyncio программе. Даже когда мы с помощью asyncio.run(...) запускали, казалось бы просто корутины, функция asyncio.run(...) “под капотом” превращала их в объекты класса asyncio.Task и после запускала в event loop.

In [ ]:
import asyncio

async def main():
  print("main started")
  await asyncio.sleep(1)
  print("main stopping...")


# Внутри функции asyncio.run корутина main сначала
# будет обёрнута в объект Task,
# который и запустится в цикле событий
asyncio.run(main())

Итак, Task — это объект, который оборачивает корутину и позволяет event loop управлять её выполнением. В стандартной asyncio-программе все Task работают конкурентно в одном потоке ОС. asyncio.Task является awaitable-объектом. То есть к нему можно применять оператор await.

Класс asyncio.Task унаследован от класса asyncio.Future. В этом случае говорят, что Task является Future-объектом.
asyncio.Future - это “низкоуровневый” класс, объект которого представляет будущий результат асинхронной операции.
Его часто называют "обещанием" (promise), ведь объект Future содержит в себе состояние асинхронной операции, которая ещё не завершена, но может завершиться (или не завершиться) в будущем.

Например, если вам надо асинхронно сделать запрос по сети, то можно создать объект Futurе, в который asyncio “обещает” поместить результат запроса, когда тот выполнится. Пока он не выполнен, соответственно, результата нет.

asyncio.Future можно ожидать при помощи await. Это будет означать, что мы ожидаем, пока Future не будет готов: то есть пока в нём не появится результат.

![иерархия](\images\awaitable_hierarchy.png)

Класс asyncio.Task же несёт в себе куда больше функционала, чем тот, что реализован в родительском asyncio.Future. Если asyncio.Future - это просто “коробка”, куда будет положен результат асинхронной операции, то asyncio.Task еще и должен:

*   Предоставить механизмы запуска себя, чтобы event loop их использовал
*   Когда event loop запустит Task, та в свою очередь должна запустить “в себе” корутину.

## Планирование Task в event loop (task scheduled)

В asyncio приложениях мы НЕ можем просто запустить корутину напрямую, как это делаем с обычными функциями. Ведь между нами и корутинами находится посредник - event loop, который сам запускает указанные нами корутины (обернутые в Task). Мы же должны запустить только сам event loop.

В то же время, мы НЕ можем сказать циклу событий: “Выполни сейчас вот эту задачу”. Мы с Вами можем лишь “зарегистрировать” задачу на выполнение в event loop, который в свою очередь запустит её, когда сам посчитает нужным. А точнее, когда у event loop “будет возможность” запустить нашу задачу.

Например, в event loop уже может "крутиться" несколько задач, и когда мы “регистрируем” новую в event loop, он поставит её в очередь, наряду с другими уже существующими задачами. И когда до неё дойдет очередь, event loop запустит и нашу задачу.

Вот эта самая “регистрация” задачи в event loop на выполнение и называется “планированием к выполнению”. То есть “запланировать задачу” (schedule task) - это всё равно, что сказать циклу событий: “Возьми задачу и начни ее выполнять, как только предоставится такая возможность”.

1.  asyncio.create_task(coro, *,name=None, context=None)  - функция из высокоуровневого API asyncio (более предпочтительный из всех).